In [2]:
import pandas as pd
import numpy as np 
import torch 
from pathlib import Path
import os 

In [3]:
dataset = Path('/kaggle/input/biggest-genderface-recognition-dataset/faces')
gender_df_model = dataset

In [4]:
gender_df_model 

PosixPath('/kaggle/input/biggest-genderface-recognition-dataset/faces')

In [5]:
gender_image = []

def convert_df(folder_Path):
    for folder_name in os.listdir(folder_Path):
        folder_path = os.path.join(folder_Path, folder_name)
        
        if os.path.isdir(folder_path):
            for image_name in os.listdir(folder_path):
                if image_name.endswith(('.png', '.jpg', '.jpeg')):  # Filter image types
                    image_path = os.path.join(folder_path, image_name)
                    if folder_Path == gender_df_model:
                        gender_image.append({'image_path': image_path, 'label': folder_name})


In [6]:
gender_df = convert_df(gender_df_model)

gender_df = pd.DataFrame(gender_image)

In [7]:
gender_df

,image_path,label
0,/kaggle/input/biggest-genderface-recognition-d...,man
1,/kaggle/input/biggest-genderface-recognition-d...,man
2,/kaggle/input/biggest-genderface-recognition-d...,man
3,/kaggle/input/biggest-genderface-recognition-d...,man
4,/kaggle/input/biggest-genderface-recognition-d...,man
...,...,...
27162,/kaggle/input/biggest-genderface-recognition-d...,woman
27163,/kaggle/input/biggest-genderface-recognition-d...,woman
27164,/kaggle/input/biggest-genderface-recognition-d...,woman
27165,/kaggle/input/biggest-genderface-recognition-d...,woman


In [8]:
gender_df.shape

(27167, 2)

In [9]:
import pandas as pd
import numpy as np
import cv2
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [10]:
def load_and_preprocess_data(gender_df, input_width, input_height):
    images = []
    labels = []
    
    for index, row in gender_df.iterrows():
        image_path = row['image_path']
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
        image = cv2.resize(image, (input_width, input_height))  # Resize
        
        images.append(image)
        labels.append(row['label'])
    
    images = np.array(images) / 255.0  # Normalize images
    images = np.expand_dims(images, axis=-1)  # Add channels dimension
    
    return images, np.array(labels)

# Example DataFrame
# emotion_df = pd.read_csv('path_to_your_data.csv')  # Uncomment and set the correct path
#emotion_df = pd.DataFrame({'image_path': ['image1.jpg', 'image2.jpg'], 
                          # 'label': ['surprised', 'neutral']})  # Replace with your actual data


In [11]:
def create_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  # Adjust for classification
    return model


In [13]:
# Preparing the dataset
input_width, input_height = 100, 100  # Modify as needed
images, labels = load_and_preprocess_data(gender_df, input_width, input_height)

# Encode labels
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded)

# Split the data
X_train, X_val, y_train, y_val = train_test_split(images, labels_categorical, test_size=0.2, random_state=42)

# Build the model
model = create_model((input_width, input_height, 1), len(le.classes_))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))


680/680 ━━━━━━━━━━━━━━━━━━━━ 152s 220ms/step - accuracy: 0.7453 - loss: 0.5018 - val_accuracy: 0.8482 - val_loss: 0.3530
Epoch 2/10
680/680 ━━━━━━━━━━━━━━━━━━━━ 149s 219ms/step - accuracy: 0.8766 - loss: 0.2957 - val_accuracy: 0.8782 - val_loss: 0.2899
Epoch 3/10
680/680 ━━━━━━━━━━━━━━━━━━━━ 150s 220ms/step - accuracy: 0.9153 - loss: 0.2138 - val_accuracy: 0.8888 - val_loss: 0.2733
Epoch 4/10
680/680 ━━━━━━━━━━━━━━━━━━━━ 148s 218ms/step - accuracy: 0.9410 - loss: 0.1501 - val_accuracy: 0.8995 - val_loss: 0.2902
Epoch 5/10
680/680 ━━━━━━━━━━━━━━━━━━━━ 150s 221ms/step - accuracy: 0.9693 - loss: 0.0865 - val_accuracy: 0.8933 - val_loss: 0.2829
Epoch 6/10
680/680 ━━━━━━━━━━━━━━━━━━━━ 149s 219ms/step - accuracy: 0.9827 - loss: 0.0523 - val_accuracy: 0.9010 - val_loss: 0.3451
Epoch 7/10
680/680 ━━━━━━━━━━━━━━━━━━━━ 147s 216ms/step - accuracy: 0.9915 - loss: 0.0284 - val_accuracy: 0.9043 - val_loss: 0.3858
Epoch 8/10
680/680 ━━━━━━━━━━━━━━━━━━━━ 151s 223ms/step - accuracy: 0.9931 - loss: 0.02

In [14]:
loss, accuracy = model.evaluate(X_val, y_val)
print(f'Validation Accuracy: {accuracy:.2f}')


170/170 ━━━━━━━━━━━━━━━━━━━━ 9s 51ms/step - accuracy: 0.8908 - loss: 0.4650
Validation Accuracy: 0.89


In [16]:
model.save('gender_model1.h5')

In [19]:
# Load your model
model = tf.keras.models.load_model('/kaggle/working/gender_model.h5')

# Print the model summary
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 98, 98, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 49, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 47, 47, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 33856)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,333,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,352,772 (16.60 MB)

 Trainable params: 4,352,770 (16.60 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [20]:
input_shape = model.input_shape
print(f"Expected input shape: {input_shape}")


Expected input shape: (None, 100, 100, 1)


In [1]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array
import random

# Define the path to the UTKFace dataset
utkface_path = '/kaggle/input/datasets/alifshahariar/utkface-dataset-face-aligned-and-labeled/utkface-aligned-labeled/images'

# Load your trained model
model = tf.keras.models.load_model('/kaggle/input/models/adityamodi20/gender-model/keras/default/1/gender_model(1).h5')

# Function to preprocess the image
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    img = cv2.resize(img, (100, 100))  # Resize to match model's input shape
    img = img.astype('float32') / 255.0  # Normalize to [0, 1]
    img = np.expand_dims(img, axis=-1)  # Add channel dimension
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    return img

# Collect all image paths and corresponding genders
image_paths = []
actual_genders = []

for filename in os.listdir(utkface_path):
    if filename.endswith('.jpg'):
        image_paths.append(os.path.join(utkface_path, filename))
        # Extract gender from filename (assuming filename format is [age]_[gender]_[race]_[filename].jpg)
        gender = int(filename.split('_')[1])  # 0 for Female, 1 for Male
        actual_genders.append(gender)

# Sample 50 random images
random_indices = random.sample(range(len(image_paths)), 50)
sample_image_paths = [image_paths[i] for i in random_indices]
sample_actual_genders = [actual_genders[i] for i in random_indices]

# Initialize counters for accuracy
correct_predictions = 0

# List to maintain predicted genders
predicted_genders = []

images_to_display = []

# Iterate through sampled images to make predictions
for image_path, actual_gender in zip(sample_image_paths, sample_actual_genders):
    img = preprocess_image(image_path)

    # Make prediction
    prediction = model.predict(img)
    predicted_gender = np.argmax(prediction)  # 0 or 1
    predicted_genders.append(predicted_gender)

    # Count correct predictions
    if predicted_gender == actual_gender:
        correct_predictions += 1
    images_to_display.append((image_path, actual_gender, predicted_gender))
# Calculate overall accuracy
accuracy = correct_predictions / len(sample_image_paths)

# Output the results
for img_path, actual, predicted in zip(sample_image_paths, sample_actual_genders, predicted_genders):
    actual_label = 'Male' if actual == 1 else 'Female'
    predicted_label = 'Male' if predicted == 1 else 'Female'
    print(f'Image: {os.path.basename(img_path)}, Actual: {actual_label}, Predicted: {predicted_label}')

print(f'\nOverall Accuracy: {accuracy * 100:.2f}%')





2026-03-19 12:38:26.249569: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773923906.485733      38 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773923906.563886      38 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-19 12:38:41.242172: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━

In [9]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array
import random
from sklearn.metrics import classification_report
import time  # Import time to measure inference time

# Define the path to the UTKFace dataset
utkface_path = '/kaggle/input/datasets/alifshahariar/utkface-dataset-face-aligned-and-labeled/utkface-aligned-labeled/images'

# Load your trained model
model = tf.keras.models.load_model('/kaggle/input/models/adityamodi20/gender-model/keras/default/1/gender_model(1).h5')

# Function to preprocess the image
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    img = cv2.resize(img, (100, 100))  # Resize to match model's input shape
    img = img.astype('float32') / 255.0  # Normalize to [0, 1]
    img = np.expand_dims(img, axis=-1)  # Add channel dimension
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    return img

# Collect all image paths and corresponding genders
image_paths = []
actual_genders = []

for  filename in os.listdir(utkface_path):
    if filename.endswith('.jpg'):
        image_paths.append(os.path.join(utkface_path, filename))
        # Extract gender from filename (assuming filename format is [age]_[gender]_[race]_[filename].jpg)
        gender = int(filename.split('_')[1])  # 0 for Female, 1 for Male
        actual_genders.append(gender)

# Sample 50 random images
random_indices = random.sample(range(len(image_paths)), 200)
sample_image_paths = [image_paths[i] for i in random_indices]
sample_actual_genders = [actual_genders[i] for i in random_indices]

# List to maintain predicted genders
predicted_genders = []

# Measure total inference time
total_inference_time = 0

# Iterate through sampled images to make predictions
for image_path, actual_gender in zip(sample_image_paths, sample_actual_genders):
    start_time = time.time()  # Start timing for each image
    img = preprocess_image(image_path)
    
    # Make prediction
    prediction = model.predict(img)
    predicted_gender = np.argmax(prediction)  # 0 or 1
    predicted_genders.append(predicted_gender)

    # Calculate inference time for the current image
    inference_time = time.time() - start_time
    
    # Format actual and predicted values
    actual_label = 'Male' if actual_gender == 1 else 'Female'
    predicted_label = 'Male' if predicted_gender == 1 else 'Female'
    
    # Print actual and predicted values with inference time
    print(f'Image: {os.path.basename(image_path)}, Actual: {actual_label}, Predicted: {predicted_label}, Inference Time: {inference_time:.4f} seconds')

# Calculate accuracy, precision, recall, and F1 score
report = classification_report(sample_actual_genders, predicted_genders, target_names=['Female', 'Male'])

# Output the classification report
print(report)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
Image: 80_1_1_20170116153903211.jpg.chip.jpg, Actual: Male, Predicted: Female, Inference Time: 0.1859 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Image: 56_0_0_20170111203105369.jpg.chip.jpg, Actual: Female, Predicted: Female, Inference Time: 0.1020 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Image: 70_0_0_20170117174910845.jpg.chip.jpg, Actual: Female, Predicted: Female, Inference Time: 0.1022 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Image: 27_0_0_20170117202216127.jpg.chip.jpg, Actual: Female, Predicted: Male, Inference Time: 0.0969 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Image: 36_1_1_20170115235941456.jpg.chip.jpg, Actual: Male, Predicted: Female, Inference Time: 0.1018 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Image: 34_0_1_20170117203210989.jpg.chip.jpg, Actual: Female, Predicted: Female, Inference Time: 0.1054 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Image: 20_1_4_20170103224540624.jpg.chip.jpg, Actual: Male, Predict

In [4]:
import os
import numpy as np
import time
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from sklearn.metrics import (
    classification_report, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# ============================================================================
# CONFIGURATION
# ============================================================================
MODEL_PATH = "/kaggle/input/models/adityamodi20/gender-model/keras/default/1/gender_model(1).h5"  # Path to your .h5 model
IMAGE_FOLDER = "/kaggle/input/models/adityamodi20/testing-image/tflite/default/1/hualcosa DLBAIPEAI01_edge_ai main evaluation_images"  # Folder containing test images
IMG_SIZE = (100, 100)  # Input size (height, width) - grayscale

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def extract_label_from_filename(filename):
    """
    Extract the true label from the image filename.
    Checks for woman/female BEFORE man/male to avoid substring matching issues.
    """
    filename_lower = filename.lower()
    
    # Check for woman FIRST (before checking for man, since "woman" contains "man")
    if "woman" in filename_lower or "female" in filename_lower:
        return "woman"
    elif "man" in filename_lower or "male" in filename_lower:
        return "man"
    else:
        return None


def load_and_preprocess_image(image_path, target_size):
    """
    Load and preprocess an image for model prediction.
    Converts to grayscale and resizes to target size.
    """
    try:
        # Load image and convert to grayscale
        img = Image.open(image_path).convert('L')  # 'L' mode = grayscale
        
        # Resize to target size
        img = img.resize(target_size, Image.Resampling.LANCZOS)
        
        # Convert to numpy array
        img_array = np.array(img)
        
        # Expand dimensions to match model input (height, width, channels)
        img_array = np.expand_dims(img_array, axis=-1)  # Add channel dimension
        img_array = np.expand_dims(img_array, axis=0)   # Add batch dimension
        
        # Normalize to [0, 1]
        img_array = img_array / 255.0
        
        return img_array
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return None


def test_model(model_path, image_folder, img_size):
    """
    Test the model on all images in a folder and generate classification metrics.
    Tracks inference time per image.
    """
    # Load the model
    print(f"Loading model from {model_path}...")
    model = load_model(model_path)
    print("Model loaded successfully!\n")
    
    # Get list of image files
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
    image_files = [
        f for f in os.listdir(image_folder)
        if os.path.splitext(f)[1].lower() in valid_extensions
    ]
    
    print(f"Found {len(image_files)} images to test.\n")
    
    # Initialize lists to store predictions and true labels
    true_labels = []
    predicted_labels = []
    predictions_raw = []
    inference_times = []  # Store inference times in milliseconds
    failed_images = []
    
    # Process each image
    for idx, filename in enumerate(image_files, 1):
        image_path = os.path.join(image_folder, filename)
        
        # Extract true label from filename
        true_label = extract_label_from_filename(filename)
        
        if true_label is None:
            print(f"[{idx}/{len(image_files)}] ⚠️  Skipping {filename} - Could not extract label from filename")
            failed_images.append(filename)
            continue
        
        # Load and preprocess image (grayscale, 100x100)
        img_array = load_and_preprocess_image(image_path, img_size)
        
        if img_array is None:
            print(f"[{idx}/{len(image_files)}] ❌ Failed to load {filename}")
            failed_images.append(filename)
            continue
        
        # Make prediction and measure inference time
        try:
            # Record start time
            start_time = time.time()
            
            # Make prediction
            prediction = model.predict(img_array, verbose=0)
            
            # Record end time and calculate elapsed time in milliseconds
            end_time = time.time()
            inference_time_ms = (end_time - start_time) * 1000
            
            predictions_raw.append(prediction[0])
            inference_times.append(inference_time_ms)
            
            # Assuming binary classification: [0] = man, [1] = woman
            # Adjust this based on your model's output
            predicted_label = "woman" if prediction[0][1] > 0.5 else "man"
            
            true_labels.append(true_label)
            predicted_labels.append(predicted_label)
            
            match = "✅" if predicted_label == true_label else "❌"
            print(f"[{idx}/{len(image_files)}] {match} {filename} | True: {true_label} | Pred: {predicted_label} | Time: {inference_time_ms:.2f}ms")
            
        except Exception as e:
            print(f"[{idx}/{len(image_files)}] ❌ Error predicting on {filename}: {e}")
            failed_images.append(filename)
    
    print(f"\n{'='*70}")
    print(f"Processing complete. Failed images: {len(failed_images)}")
    print(f"Successfully processed: {len(true_labels)} images\n")
    
    if failed_images:
        print("Failed images:")
        for img in failed_images:
            print(f"  - {img}")
        print()
    
    return true_labels, predicted_labels, predictions_raw, inference_times


def generate_report(true_labels, predicted_labels, inference_times):
    """
    Generate comprehensive classification metrics and report.
    Includes inference time statistics.
    """
    print(f"{'='*70}")
    print("CLASSIFICATION REPORT")
    print(f"{'='*70}\n")
    
    # Overall Accuracy
    accuracy = accuracy_score(true_labels, predicted_labels)
    print(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)\n")
    
    # Detailed Classification Report
    print("Detailed Classification Report:")
    print("-" * 70)
    report = classification_report(
        true_labels, 
        predicted_labels,
        target_names=['man', 'woman'],
        digits=4
    )
    print(report)
    
   
    # Confusion Matrix
    print(f"{'='*70}")
    print("CONFUSION MATRIX")
    print(f"{'='*70}\n")
    cm = confusion_matrix(true_labels, predicted_labels, labels=['man', 'woman'])
    print("                 Predicted")
    print("                 man  woman")
    print(f"Actual man      {cm[0][0]:3d}  {cm[0][1]:3d}")
    print(f"       woman    {cm[1][0]:3d}  {cm[1][1]:3d}\n")
    
    # Inference Time Statistics
    print(f"{'='*70}")
    print("INFERENCE TIME STATISTICS")
    print(f"{'='*70}\n")
    
    avg_inference_time = np.mean(inference_times)
    min_inference_time = np.min(inference_times)
    max_inference_time = np.max(inference_times)
    std_inference_time = np.std(inference_times)
    
    print(f"Average Inference Time: {avg_inference_time:.2f} ms")
    print(f"Min Inference Time:     {min_inference_time:.2f} ms")
    print(f"Max Inference Time:     {max_inference_time:.2f} ms")
    print(f"Std Dev Inference Time: {std_inference_time:.2f} ms")
    print(f"Total Images Processed: {len(inference_times)}")
    print(f"Total Processing Time:  {sum(inference_times):.2f} ms ({sum(inference_times)/1000:.2f} seconds)\n")
    
    return accuracy, precision_macro, recall_macro, f1_macro, cm, avg_inference_time, min_inference_time, max_inference_time, std_inference_time





# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Test the model
    true_labels, predicted_labels, predictions_raw, inference_times = test_model(
        MODEL_PATH, 
        IMAGE_FOLDER, 
        IMG_SIZE
    )
    
    # Generate report
    if len(true_labels) > 0:
        accuracy, precision, recall, f1, cm, avg_time, min_time, max_time, std_time = generate_report(
            true_labels, 
            predicted_labels, 
            inference_times
        )
        
        
        # Save results to file
        with open('test_results.txt', 'w') as f:
            f.write("GENDER CLASSIFICATION MODEL TEST RESULTS\n")
            f.write("=" * 70 + "\n\n")
            f.write(f"Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)\n")
            f.write(f"Macro-averaged Precision: {precision:.4f}\n")
            f.write(f"Macro-averaged Recall: {recall:.4f}\n")
            f.write(f"Macro-averaged F1-Score: {f1:.4f}\n\n")
           
          
            
        
        print("Results saved to 'test_results.txt'")
    else:
        print("No images were successfully processed. Please check your configuration.")


Loading model from /kaggle/input/models/adityamodi20/gender-model/keras/default/1/gender_model(1).h5...
Model loaded successfully!

Found 24 images to test.

[1/24] ✅ young_woman_sad_3.jpg | True: woman | Pred: woman | Time: 122.95ms
[2/24] ✅ young_woman_happy_2.jpg | True: woman | Pred: woman | Time: 73.03ms
[3/24] ✅ elderly_woman_happy_3.jpg | True: woman | Pred: woman | Time: 69.13ms
[4/24] ✅ young_man_happy_3.jpg | True: man | Pred: man | Time: 79.43ms
[5/24] ✅ young_man_sad_1.jpg | True: man | Pred: man | Time: 78.93ms
[6/24] ✅ elderly_man_happy_1.jpg | True: man | Pred: man | Time: 79.32ms
[7/24] ✅ young_man_sad_2.jpg | True: man | Pred: man | Time: 73.76ms
[8/24] ✅ young_woman_sad_1.jpg | True: woman | Pred: woman | Time: 76.11ms
[9/24] ❌ elderly_man_sad_3.jpg | True: man | Pred: woman | Time: 70.91ms
[10/24] ✅ elderly_man_happy_2.jpg | True: man | Pred: man | Time: 80.17ms
[11/24] ✅ elderly_man_happy_3.jpg | True: man | Pred: man | Time: 74.51ms
[12/24] ✅ young_man_happy_1.jpg 